# Convols: Build the Multiresolution Field

This notebook shows how PyHermes converts particle data into `ConvolsData`, the multiresolution field representation used by later tasks such as counting, 2PCF, and 3PCF.

## What this notebook emphasizes

- how supported particle readers behave on real inputs
- how to run `Convols` from progressively lower-level interfaces
- how to convert a velocity catalog from real space to redshift space before building the field
- how to build a matching random field for later `DR` / `RR` calculations
- how window convolution is applied once the field has been built

## Suggested reading order

A typical walkthrough is:

1. `convols.ipynb` to build the field
2. `window.ipynb` to learn field/window algebra and smoothing filters
3. `counting.ipynb` to sample the field at random points
4. `corr2pcf.ipynb` to measure two-point statistics
5. `corr3pcf.ipynb` to measure three-point statistics


In [1]:
import numpy as np
import yaml
from pyhermes.base.convols import Convols
from pyhermes.param.parambase import read_param
from pyhermes.utils.sampling import random_points_box
from pyhermes.utils.func_util import validate_convols_compatibility
from pyhermes.utils.redshift_space import hubble_at_redshift, redshift_space_positions
from pyhermes.io import ConvolsData, read_particle_data

from pathlib import Path
import os
os.chdir(Path.cwd().resolve().parent)
print(f"Working directory: {Path.cwd()}")

Working directory: /Users/xutengpeng/xutp/PycharmProjects/PyHermes/examples


## Optional: download the example halo catalog

If `./data/quijote_halos/` is not available locally, run the next cell to download the archive from the PyHermes documentation site into `examples/data/` and extract it in place.


In [2]:
!mkdir -p ./data
!curl -L \
  -A 'Mozilla/5.0' \
  -e 'https://pyhermes.astroslacker.com/' \
  -o ./data/quijote_halos.tar.gz \
  'https://pyhermes.astroslacker.com/_downloads/87691d6e7eb8dd0b954576b2bc71fb51/quijote_halos.tar.gz'

!tar -xzf ./data/quijote_halos.tar.gz -C ./data --exclude='._*' --exclude='__MACOSX'


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9554k  100 9554k    0     0  4122k      0  0:00:02  0:00:02 --:--:-- 4123k


## 1. Read particle data directly

The next few cells are reader checks. They show that PyHermes can ingest the same dataset through different entry points before any multiresolution field is constructed.


In [3]:
reader_params = {
    "snapnum": 4,
    "redshift": 0.0,
    "fields": {
        "vel": "vel",
        "vx": "vel_x",
        "vy": "vel_y",
        "vz": "vel_z",
        "mass": "mass",
        "npart": "npart",
    }
}
fof_data = read_particle_data("./data/quijote_halos/8000", data_format="fof", **reader_params)
fof_data


11:46:59 - INFO - pyhermes.io.readers:read_particle_data - Selected input particle format: fof
11:46:59 - INFO - pyhermes.io.readers:read_fof - Reading Quijote FoF halo data from ---> ./data/quijote_halos/8000 <---


{'pos': array([[200.74644 , 133.12454 ,  26.963882],
        [925.44214 ,  71.67172 , 696.62683 ],
        [954.62994 , 828.08026 , 177.58418 ],
        ...,
        [999.9344  , 998.4424  , 985.7499  ],
        [998.38574 , 933.26776 , 903.9426  ],
        [998.53217 , 972.2057  , 925.5271  ]], dtype=float32),
 'size': 406728,
 'vel': array([[-207.89943 ,  118.602196,   85.94662 ],
        [-488.92133 ,  336.5252  ,  314.52228 ],
        [  15.307102,   76.373116, -266.05084 ],
        ...,
        [ 109.77485 ,   -9.714653, -321.05997 ],
        [  27.979118,  415.54718 , 1027.4452  ],
        [ -11.882952,  -58.924763,  131.56258 ]], dtype=float32),
 'vx': array([-207.89943 , -488.92133 ,   15.307102, ...,  109.77485 ,
          27.979118,  -11.882952], dtype=float32),
 'vy': array([118.602196, 336.5252  ,  76.373116, ...,  -9.714653, 415.54718 ,
        -58.924763], dtype=float32),
 'vz': array([  85.94662,  314.52228, -266.05084, ..., -321.05997, 1027.4452 ,
         131.56258], d

### Optional: Repack the FoF halo catalog into the documented raw binary table

The Quijote release ships the original `group_tab` catalog, not the compact `.bin` table used in some PyHermes examples. The next cell reads `./data/quijote_halos/quijote_halo_bin_schema.yaml`, follows its documented column order, and writes a dense float32 table that can be consumed by the generic `bin` reader.


In [4]:


schema_path = Path("./data/quijote_halos/quijote_halo_bin_schema.yaml")
bin_path = Path("./data/quijote_halos/8000/groups_004/group_tab_004.bin")

schema = yaml.safe_load(schema_path.read_text())
columns = schema["columns"]

column_arrays = {
    "x": fof_data["pos"][:, 0],
    "y": fof_data["pos"][:, 1],
    "z": fof_data["pos"][:, 2],
    "vx": fof_data["vx"],
    "vy": fof_data["vy"],
    "vz": fof_data["vz"],
    "mass": fof_data["mass"],
    "npart": fof_data["npart"],
}

missing = [name for name in columns if name not in column_arrays]
if missing:
    raise KeyError(f"Cannot build the binary table because these schema columns are missing: {missing}")

table = np.column_stack([
    np.asarray(column_arrays[name], dtype=np.float32)
    for name in columns
]).astype(np.float32, copy=False)

if table.shape[1] != schema["ncols"]:
    raise ValueError(
        f"Schema expects {schema['ncols']} columns, but the packed table has shape {table.shape}."
    )

bin_path.parent.mkdir(parents=True, exist_ok=True)
table.tofile(bin_path)
print(f"Wrote {bin_path} with shape {table.shape} and dtype {table.dtype}.")


Wrote data/quijote_halos/8000/groups_004/group_tab_004.bin with shape (406728, 8) and dtype float32.


In [5]:
reader_params = {
    "dtype": schema["dtype"],
    "ncols": schema["ncols"],
    "pos_cols": [0, 1, 2],
    "fields": {
        "vel": [3, 4, 5],
        "vx": 3,
        "vy": 4,
        "vz": 5,
        "mass": 6,
        "npart": 7,
    }
}
bin_data = read_particle_data(str(bin_path), **reader_params)
{
    "size_match": bin_data["size"] == fof_data["size"],
    "pos_match": np.allclose(bin_data["pos"], fof_data["pos"]),
    "vel_match": np.allclose(bin_data["vel"], fof_data["vel"]),
    "mass_match": np.allclose(bin_data["mass"], fof_data["mass"]),
    "npart_match": np.array_equal(bin_data["npart"], fof_data["npart"].astype(np.float32)),
}


11:46:59 - INFO - pyhermes.io.readers:read_particle_data - Selected input particle format: bin
11:46:59 - INFO - pyhermes.io.readers:read_bin - Reading particle data from ---> data/quijote_halos/8000/groups_004/group_tab_004.bin <---


{'size_match': True,
 'pos_match': True,
 'vel_match': True,
 'mass_match': True,
 'npart_match': True}

## 2. Build `ConvolsData`

`Convols` is the upstream stage of the whole workflow. It takes particle positions, projects them into the PyHermes multiresolution basis, and returns a field object that can later be sampled, convolved, or correlated.

### Core idea

- `box_size` sets the physical domain size.
- `J` controls the multiresolution level.
- `wavelet_mode`, `wavelet_level`, and `phi_resolution` control the basis construction.
- the output `ConvolsData` object stores both the field values and the metadata required by later window operations.


### Minimal YAML Shapes

`Convols` builds the saved multiresolution field used by the later notebooks. A minimal config specifies the input reader, the box/grid parameters, the wavelet basis, and the output path:

```yaml
Convols:
   fin:
      path: "./data/quijote_halos/8000"
      format: "fof"
      reader_params:
         snapnum: 4
   box_size: 1000
   J: 8
   wavelet_mode: "db2"
   wavelet_level: 10
   phi_resolution: 1024
   threads: 2
   fout_path: "./output/quijote8000_snap004_sfc.pkl"
```

Optional fields such as `weight_key`, `save_particle_data`, and `particle_data_path` control how particle weights and companion particle arrays are handled. Setting `weight_key: "mass"` builds the mass-weighted version of the same catalog. The examples below start from this shape and then add checks for direct particle reading, mass weighting, redshift-space conversion, and matching random fields.


### Command-line entry point

This is the most direct way to run `Convols` in production. A YAML file defines the task, and the standard driver script handles the run.


In [6]:
! mpirun -np 4 python ./scripts/run_convols.py ./configs/param_convols.yaml

11:47:00 - INFO - pyhermes.param.parambase:ParamBase - Reading configure file: './configs/param_convols.yaml'
11:47:00 - INFO - pyhermes.param.parambase:ParamBase - Input parameter file format is YAML
11:47:00 - INFO - pyhermes.param.parambase:ParamBase - Set default parameters of module <base> ...
11:47:00 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.fin.path' from 'empty' to './data/quijote_halos/8000'
11:47:00 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.fin.format' from 'bin' to 'fof'
11:47:00 - WARNING - pyhermes.param.parambase:ParamBase - Adding non-default key: 'Convols.fin.reader_params.snapnum'
11:47:00 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.threads' from '1' to '2'
11:47:00 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.save_particle_data' from 'False' to 'True'
11:47:00 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.particle_data_path' from 'empty' to './data/quijote_halos/8000/group

### Config-driven Python API

This version still treats the YAML config as the main source of truth, but launches the task from Python so that you can inspect objects in the notebook.


In [7]:
convols_params = read_param(config_path="./configs/param_convols.yaml")
task = Convols(param_task=convols_params)
task.threads = 8
task.save_particle_data = False
convols_data = task.run(save_result=False)

11:47:02 - INFO - pyhermes.param.parambase:ParamBase - Reading configure file: './configs/param_convols.yaml'
11:47:02 - INFO - pyhermes.param.parambase:ParamBase - Input parameter file format is YAML
11:47:02 - INFO - pyhermes.param.parambase:ParamBase - Set default parameters of module <base> ...
11:47:02 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.fin.path' from 'empty' to './data/quijote_halos/8000'
11:47:02 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.fin.format' from 'bin' to 'fof'
11:47:02 - WARNING - pyhermes.param.parambase:ParamBase - Adding non-default key: 'Convols.fin.reader_params.snapnum'
11:47:02 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.threads' from '1' to '2'
11:47:02 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.save_particle_data' from 'False' to 'True'
11:47:02 - INFO - pyhermes.param.parambase:ParamBase - Default 'Convols.particle_data_path' from 'empty' to './data/quijote_halos/8000/group

### Task object overrides

Here the task object is created first and then adjusted in Python. This is useful when the config is close to what you want, but a few runtime settings need to change.

The next two cells intentionally create two independent `Convols()` objects: one for a higher-resolution `J=9` field, and one for a mass-weighted field. Treat each saved field as its own task run. If particle inputs, weights, grid parameters, or reader settings change, start from a fresh task object instead of reusing a previously prepared one.


In [8]:
task = Convols()
task.fin = {
    "path": "./data/quijote_halos/8000",
    "format": "fof",
    "reader_params": {
        "snapnum": 4
    }
}
task.J = 9
task.threads = 8
task.fout_path = "./output/quijote8000_snap004_sfc_J9.pkl"
task.run(overwrite=True)

11:47:03 - INFO - pyhermes.param.parambase:ParamBase - Set default parameters of module <base> ...
11:47:03 - INFO - pyhermes.pipeline.pipeline:Convols - Convols runtime configuration: running on 1 MPI ranks with 8 threads per rank
11:47:03 - INFO - pyhermes.pipeline.pipeline:Convols - Preparing Convols input fields ...
11:47:03 - INFO - pyhermes.pipeline.pipeline:Convols - J=9, L=512, box_size=1000, phi_resolution=1024, wavelet_mode=db2, wavelet_level=10
11:47:03 - INFO - pyhermes.io.readers:read_particle_data - Selected input particle format: fof
11:47:03 - INFO - pyhermes.io.readers:read_fof - Reading Quijote FoF halo data from ---> ./data/quijote_halos/8000 <---
11:47:03 - INFO - pyhermes.pipeline.pipeline:Convols - Input particles ready | source=file=path=./data/quijote_halos/8000 format=fof | particle_count=406728 | weight_key=None | weight_sum=4.067280e+05
11:47:03 - INFO - pyhermes.pipeline.pipeline:Convols - Single process mode
11:47:03 - INFO - pyhermes.pipeline.pipeline:Conv

In [9]:
task = Convols()
task.fin = {
    "path": "./data/quijote_halos/8000",
    "format": "fof",
    "reader_params": {
        "snapnum": 4,
        "fields": {
            "mass": "mass",
        }
    },
    "weight_key": "mass"
}
task.threads = 8
task.fout_path = "./output/quijote8000_snap004_sfc_massweight.pkl"
task.run(overwrite=True)

11:47:05 - INFO - pyhermes.param.parambase:ParamBase - Set default parameters of module <base> ...
11:47:05 - INFO - pyhermes.pipeline.pipeline:Convols - Convols runtime configuration: running on 1 MPI ranks with 8 threads per rank
11:47:05 - INFO - pyhermes.pipeline.pipeline:Convols - Preparing Convols input fields ...
11:47:05 - INFO - pyhermes.pipeline.pipeline:Convols - J=8, L=256, box_size=1000, phi_resolution=1024, wavelet_mode=db2, wavelet_level=10
11:47:05 - INFO - pyhermes.io.readers:read_particle_data - Selected input particle format: fof
11:47:05 - INFO - pyhermes.io.readers:read_fof - Reading Quijote FoF halo data from ---> ./data/quijote_halos/8000 <---
11:47:05 - INFO - pyhermes.pipeline.pipeline:Convols - Input particles ready | source=file=path=./data/quijote_halos/8000 format=fof | particle_count=406728 | weight_key=mass | weight_sum=1.964693e+19
11:47:05 - INFO - pyhermes.pipeline.pipeline:Convols - Single process mode
11:47:05 - INFO - pyhermes.pipeline.pipeline:Conv

### Manual particle input and redshift-space preparation

At this level the particle positions are prepared explicitly and injected into the task object. This is the most flexible route when your upstream data preparation is custom.

It is also the most natural place to insert a real-space to redshift-space mapping: load positions and velocities, shift the particles along a chosen line of sight, and then run `Convols` on the transformed catalog.

As above, each output is built with a fresh `Convols()` instance. This keeps the prepared particle arrays, weight normalization, and metadata tied to exactly one saved field.


In [10]:
reader_params = {
    "snapnum": 4,
    "redshift": 0.0,
    "fields": {
        "vel": "vel",
        "mass": "mass",
    }
}
data = read_particle_data("./data/quijote_halos/8000", data_format="fof", **reader_params)
hubble_parameter = hubble_at_redshift(redshift=0)

11:47:05 - INFO - pyhermes.io.readers:read_particle_data - Selected input particle format: fof
11:47:05 - INFO - pyhermes.io.readers:read_fof - Reading Quijote FoF halo data from ---> ./data/quijote_halos/8000 <---


#### Build redshift-space fields for different lines of sight

The next cells use `redshift_space_positions(...)` to transform the same halo sample into redshift space and then build `ConvolsData` from the shifted positions.

- first with `los="z"` and unit weights for a box-axis line of sight
- then with the same z-axis redshift-space positions but mass weights
- finally with `los=[1, 1, 1]` for a diagonal line of sight

These outputs are useful downstream because anisotropic statistics such as `(s, mu)` 2PCF are sensitive to both the presence of redshift-space distortions and the chosen viewing direction. The mass-weighted output gives a parallel field for checking how tracer weights change later measurements.


In [11]:
pos = redshift_space_positions(data['pos'], data['vel'], box_size=1000,
                               hubble=hubble_parameter, redshift=0, los="z")
task = Convols()
task.particle_pos = pos
task.threads = 8
task.fout_path = "./output/quijote8000_snap004_rsd_sfc.pkl"
task.run(overwrite=True)

11:47:06 - INFO - pyhermes.param.parambase:ParamBase - Set default parameters of module <base> ...
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Convols runtime configuration: running on 1 MPI ranks with 8 threads per rank
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Preparing Convols input fields ...
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - J=8, L=256, box_size=1000, phi_resolution=1024, wavelet_mode=db2, wavelet_level=10
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - No particle_weight provided; using unit weights for 406728 particles.
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Input particles ready | source=custom particle_pos array | particle_count=406728 | weight_key=None | weight_sum=4.067280e+05
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Single process mode
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - The time for scaling function: 0.1782 sec
11:47:06 - WARNING - pyhermes.io.funcs:check_fout - Output file

In [12]:
task = Convols()
task.particle_pos = pos
task.particle_weight = data["mass"]
task.threads = 8
task.fout_path = "./output/quijote8000_snap004_rsd_sfc_massweight.pkl"
task.run(overwrite=True)

11:47:06 - INFO - pyhermes.param.parambase:ParamBase - Set default parameters of module <base> ...
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Convols runtime configuration: running on 1 MPI ranks with 8 threads per rank
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Preparing Convols input fields ...
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - J=8, L=256, box_size=1000, phi_resolution=1024, wavelet_mode=db2, wavelet_level=10
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Input particles ready | source=custom particle_pos array | particle_count=406728 | weight_key=custom | weight_sum=1.964693e+19
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Single process mode
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - The time for scaling function: 0.1237 sec
11:47:06 - INFO - pyhermes.io.base:ConvolsData - Writing Convols data to ---> ./output/quijote8000_snap004_rsd_sfc_massweight.pkl <---

11:47:06 - INFO - pyhermes.pipeline.pipeline:Convo

In [13]:
pos = redshift_space_positions(data['pos'], data['vel'], box_size=1000,
                               hubble=hubble_parameter, redshift=0, los=[1, 1, 1])
task = Convols()
task.particle_pos = pos
task.threads = 8
task.fout_path = "./output/quijote8000_snap004_rsd_diag_sfc.pkl"
task.run(overwrite=True)

11:47:06 - INFO - pyhermes.param.parambase:ParamBase - Set default parameters of module <base> ...
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Convols runtime configuration: running on 1 MPI ranks with 8 threads per rank
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Preparing Convols input fields ...
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - J=8, L=256, box_size=1000, phi_resolution=1024, wavelet_mode=db2, wavelet_level=10
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - No particle_weight provided; using unit weights for 406728 particles.
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Input particles ready | source=custom particle_pos array | particle_count=406728 | weight_key=None | weight_sum=4.067280e+05
11:47:06 - INFO - pyhermes.pipeline.pipeline:Convols - Single process mode
11:47:07 - INFO - pyhermes.pipeline.pipeline:Convols - The time for scaling function: 0.1494 sec
11:47:07 - WARNING - pyhermes.io.funcs:check_fout - Output file

## 3. Build a matching random field

Many downstream estimators compare the data field to a random field with the same grid and metadata. The next cell constructs such a random catalog and stores it as a second `ConvolsData` object.


In [14]:
random_pos = random_points_box(N=10_000_000, box_size=1000, seed=42)
task = Convols()
task.particle_pos = random_pos
task.fout_path = "./output/random_sfc.pkl"
task.save_particle_data = True
task.particle_data_path = "./data/random_1e7.npz"
task.threads = 8
task.run(overwrite=True)

11:47:07 - INFO - pyhermes.param.parambase:ParamBase - Set default parameters of module <base> ...
11:47:07 - INFO - pyhermes.pipeline.pipeline:Convols - Convols runtime configuration: running on 1 MPI ranks with 8 threads per rank
11:47:07 - INFO - pyhermes.pipeline.pipeline:Convols - Preparing Convols input fields ...
11:47:07 - INFO - pyhermes.pipeline.pipeline:Convols - J=8, L=256, box_size=1000, phi_resolution=1024, wavelet_mode=db2, wavelet_level=10
11:47:07 - INFO - pyhermes.pipeline.pipeline:Convols - No particle_weight provided; using unit weights for 10000000 particles.
11:47:07 - INFO - pyhermes.pipeline.pipeline:Convols - Input particles ready | source=custom particle_pos array | particle_count=10000000 | weight_key=None | weight_sum=1.000000e+07
11:47:07 - INFO - pyhermes.pipeline.pipeline:Convols - Saved particle positions and weights to ./data/random_1e7.npz
11:47:07 - INFO - pyhermes.pipeline.pipeline:Convols - Single process mode
11:47:11 - INFO - pyhermes.pipeline.pip

## 4. Reload the data and random fields

Once both fields exist on disk, PyHermes can reload them and verify that their required metadata are compatible. This compatibility check matters because subtraction, pair products, and later correlation estimators assume that both fields live on the same grid.


In [15]:
D = ConvolsData(data_path='./output/quijote8000_snap004_sfc.pkl', threads=8)
R = ConvolsData(data_path='./output/random_sfc.pkl', threads=8)

11:47:11 - INFO - pyhermes.io.base:ConvolsData - Reading Convols data from ---> ./output/quijote8000_snap004_sfc.pkl <---
11:47:11 - INFO - pyhermes.io.base:ConvolsData - epsilon: Shape(256, 256, 256), Min = -3.215e-06, Max = 1.343e-05, Mean = 5.96e-08, Sum = 1
11:47:11 - INFO - pyhermes.io.base:ConvolsData - Reading Convols data from ---> ./output/random_sfc.pkl <---
11:47:11 - INFO - pyhermes.io.base:ConvolsData - epsilon: Shape(256, 256, 256), Min = -2.192e-07, Max = 9.48e-07, Mean = 5.96e-08, Sum = 1


In [16]:
shared_required = validate_convols_compatibility([D, R], ConvolsData._REQUIRED_ARGV)
print("Compatibility check passed. Shared required parameters:")
print(", ".join([f"{k}={v}" for k, v in shared_required.items()]))

Compatibility check passed. Shared required parameters:
J=8, box_size=1000, phi_resolution=1024, wavelet_mode=db2, wavelet_level=10
